# Notebook 19
## Protein ProtBERT Embedding Extraction

Extracts pretrained transformer embeddings for protein sequences using
**ProtBERT** (`Rostlab/prot_bert`), a BERT-large model pretrained on
~217 million UniRef100 protein sequences via masked language modelling.

### Pipeline
```
Protein sequence (variable length, max 1024 aa)
-> ProtBERT tokenizer  (space-separated single amino acids)
-> BERT encoder (29 layers, 1024 hidden dim, ~420M parameters)
-> Mean pooling over body tokens (CLS and SEP excluded)
-> Fixed-length embedding vector (1024-dim)
-> Save as .npy
```

### Key differences from ESM-2 (notebook 13)
- ESM-2 used the `esm` library with its own batch converter and layer extraction.
  ProtBERT uses standard HuggingFace `AutoTokenizer` / `AutoModel`.
- ESM-2 excluded CLS/EOS by index slicing (`[1:seq_len+1]`).
  ProtBERT uses `[CLS]` at position 0 and `[SEP]` at the final real token + 1.
  We zero out both in the attention mask before mean pooling.
- ProtBERT requires sequences to be **space-separated** at the amino acid level:
  `'MKTL...'` must become `'M K T L ...'` before tokenization.
- ProtBERT represents rare/ambiguous amino acids (B, Z, U, O, X) as `[UNK]`.
  These are present in some UniProt sequences; we keep them as-is.

### Outputs
- `data/processed/protein_protbert_embeddings_top10_per400.npy`
- `data/processed/protein_protbert_labels_top10_per400.npy`
- `data/processed/protein_protbert_family_names_top10_per400.npy`
- `reports/protein_protbert_embeddings_summary.json`

## 0) Installation note

```bash
pip install transformers torch numpy pandas pyyaml tqdm sentencepiece
```

Model card: https://huggingface.co/Rostlab/prot_bert

No `trust_remote_code=True` needed.
ProtBERT uses standard HuggingFace BertModel registered in transformers core.

## 1) Imports

In [1]:
import json
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')

print('torch:          ', torch.__version__)
print('CUDA available: ', torch.cuda.is_available())


torch:           2.10.0+cu128
CUDA available:  True


## 2) Paths, config, seed

In [2]:
ROOT      = Path.cwd().parents[0]
PROCESSED = ROOT / 'data' / 'processed'
REPORTS   = ROOT / 'reports'
CONFIGS   = ROOT / 'configs'

for p in [PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

with open(CONFIGS / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

SEED    = int(cfg['project']['random_seed'])
N_FAM   = int(cfg['protein']['n_families'])
PER_FAM = int(cfg['protein']['per_family'])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'SEED={SEED}  N_FAM={N_FAM}  PER_FAM={PER_FAM}')


SEED=42  N_FAM=10  PER_FAM=400


## 3) Device selection

In [3]:
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print('Using device:', DEVICE)


Using device: cuda


## 4) Load protein dataset

Same CSV used throughout the protein pipeline (notebooks 01, 09-14).
All rows are embedded in original order; notebook 20 applies the split.

In [4]:
in_csv = PROCESSED / f'protein_uniprot_pfam_top{N_FAM}_per{PER_FAM}.csv'
assert in_csv.exists(), f'Not found: {in_csv} -- run 01_protein_ingest.ipynb first'

df = pd.read_csv(in_csv)

required = ['accession', 'sequence', 'family']
assert all(c in df.columns for c in required)
assert df['sequence'].isna().sum() == 0
assert df['family'].isna().sum() == 0

print('Shape:', df.shape)
print('Families:', df['family'].nunique())
print(df['family'].value_counts().to_string())
print(df.head(3))


Shape: (2293, 4)
Families: 10
family
PF13853    400
PF01352    350
PF07686    309
PF00001    287
PF00069    203
PF00096    194
PF00046    187
PF00071    125
PF00076    121
PF12796    117
  accession                                           sequence  length  \
0    Q96RD1  MRNHTEITEFILLGLTDDPNFQVVIFVFLLITYMLSITGNLTLITI...     312   
1    Q9H210  MRQINQTQVTEFLLLGLSDGPHTEQLLFIVLLGVYLVTVLGNLLLI...     308   
2    Q8NGZ3  MNHSVVTEFIILGLTKKPELQGIIFLFFLIVYLVAFLGNMLIIIAK...     307   

    family  
0  PF13853  
1  PF13853  
2  PF13853  


## 5) Load ProtBERT tokenizer and model

### Tokenizer requirement: space-separated amino acids
ProtBERT's vocabulary is built on single amino acids separated by spaces.
The sequence `'MKTLL'` must be passed as `'M K T L L'`.
This is handled in the preprocessing function below -- do not pass raw
sequences directly to the tokenizer.

### Memory
ProtBERT (~420M parameters, BERT-large architecture) requires ~1.7 GB GPU
memory at fp32. For variable-length protein sequences (up to 1024 aa),
`batch_size=8` is safe on a 16 GB GPU. Use 16 on A100.

In [5]:
MODEL_NAME = 'Rostlab/prot_bert'

print(f'Loading tokenizer from: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
print(f'  vocab_size:   {tokenizer.vocab_size}')
print(f'  cls_token:    {tokenizer.cls_token}  id={tokenizer.cls_token_id}')
print(f'  sep_token:    {tokenizer.sep_token}  id={tokenizer.sep_token_id}')
print(f'  pad_token:    {tokenizer.pad_token}  id={tokenizer.pad_token_id}')

print(f'\nLoading model from: {MODEL_NAME}')
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Model dtype:      {next(model.parameters()).dtype}')


Loading tokenizer from: Rostlab/prot_bert


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 419,931,136
Model dtype:      torch.float32


## 6) Tokenization check

Verify space-separated formatting and token structure for one example.
A 300 aa sequence should produce 300 body tokens + CLS + SEP = 302 tokens.

In [6]:
def space_separate(seq: str) -> str:
    """Convert 'MKTLL' -> 'M K T L L' as required by ProtBERT."""
    # Also replace rare amino acids that ProtBERT maps to UNK
    seq = re.sub(r'[UZOB]', 'X', seq.upper())
    return ' '.join(list(seq))


example_seq = df['sequence'].iloc[0]
example_fmt = space_separate(example_seq)

enc = tokenizer(
    example_fmt,
    return_tensors='pt',
    padding=False,
    truncation=True,
    max_length=1024,
)

tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].tolist())
print(f'Sequence length (aa):    {len(example_seq)}')
print(f'input_ids shape:         {enc["input_ids"].shape}')
print(f'attention_mask sum:      {enc["attention_mask"].sum().item()}')
print(f'Token 0 (CLS):           {tokens[0]}')
print(f'Last token (SEP):        {tokens[-1]}')
print(f'First 10 body tokens:    {tokens[1:11]}')


Sequence length (aa):    312
input_ids shape:         torch.Size([1, 314])
attention_mask sum:      314
Token 0 (CLS):           [CLS]
Last token (SEP):        [SEP]
First 10 body tokens:    ['M', 'R', 'N', 'H', 'T', 'E', 'I', 'T', 'E', 'F']


## 7) Embedding extraction function

### Pooling strategy
We exclude both `[CLS]` (position 0) and `[SEP]` (last real token) from the
mean pool, averaging only over the amino acid body tokens.

This is done by zeroing positions 0 and -1 (last non-padding position) in
the attention mask before computing the weighted mean. The `[SEP]` token is
always the last real token in ProtBERT's tokenisation scheme.

Concretely, for sequence of length L (aa):
```
token layout: [CLS] aa_1 aa_2 ... aa_L [SEP] [PAD] [PAD] ...
mask:           1     1    1  ...  1     1     0     0   ...
after zeroing:  0     1    1  ...  1     0     0     0   ...
                ^CLS zeroed           ^SEP zeroed
pool over:            aa_1 ... aa_L
```

### Sequence preprocessing
Each sequence is space-separated and rare amino acids (U, Z, O, B) are
replaced with X before tokenization.

In [7]:
def mean_pool_body(hidden_states, attention_mask):
    """
    Mean pool over amino acid body tokens, excluding CLS (pos 0) and SEP
    (last real token per sequence).

    Args:
        hidden_states:  (batch, seq_len, hidden_dim)
        attention_mask: (batch, seq_len)  -- 1 real, 0 padding

    Returns:
        pooled: (batch, hidden_dim)
    """
    mask = attention_mask.clone().float()   # (B, T)

    # Zero CLS at position 0
    mask[:, 0] = 0.0

    # Zero SEP: last position where mask == 1 in each sequence
    # sep_idx[i] = index of last real token (SEP) in sequence i
    seq_lengths = attention_mask.sum(dim=1)  # (B,)  total real tokens incl CLS+SEP
    for i in range(mask.shape[0]):
        sep_idx = int(seq_lengths[i].item()) - 1
        mask[i, sep_idx] = 0.0

    mask_expanded = mask.unsqueeze(-1)                          # (B, T, 1)
    sum_hidden    = (hidden_states * mask_expanded).sum(dim=1)  # (B, D)
    sum_mask      = mask_expanded.sum(dim=1).clamp(min=1e-9)    # (B, 1)
    return sum_hidden / sum_mask                                 # (B, D)


def extract_protbert_embeddings(
    sequences,
    tokenizer,
    model,
    device,
    batch_size=8,
    max_length=1024,
):
    """
    Extract ProtBERT mean-pooled embeddings (CLS and SEP excluded).

    Args:
        sequences:   List of raw protein sequences (no space-separation needed;
                     handled internally).
        tokenizer:   HuggingFace tokenizer for ProtBERT.
        model:       Loaded ProtBERT AutoModel in eval mode.
        device:      Torch device string.
        batch_size:  Sequences per forward pass (8 safe on 16 GB GPU).
        max_length:  Hard token limit (1024 matches ProtBERT pretraining limit).

    Returns:
        embeddings: np.ndarray of shape (len(sequences), 1024).
    """
    all_embeddings = []
    model.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), batch_size),
                      desc='Extracting ProtBERT embeddings'):
            batch_seqs = sequences[i : i + batch_size]

            # Space-separate and replace rare amino acids
            batch_fmt = [space_separate(s) for s in batch_seqs]

            encoded = tokenizer(
                batch_fmt,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            input_ids      = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            hidden_states = outputs.last_hidden_state  # (B, T, D)

            pooled = mean_pool_body(hidden_states, attention_mask)
            all_embeddings.append(pooled.cpu().float().numpy())

    return np.concatenate(all_embeddings, axis=0)


## 8) Run extraction

Expected runtime on A100 (40 GB) with `batch_size=16`: ~3-4 min for 2293
sequences (variable length, up to 1024 aa).
Reduce `BATCH_SIZE` if you get CUDA OOM -- longer sequences use more memory.

In [8]:
# 16 GB GPU -> 8;  40 GB GPU (A100) -> 16
BATCH_SIZE = 8

sequences = df['sequence'].tolist()

embeddings = extract_protbert_embeddings(
    sequences=sequences,
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    max_length=1024,
)

print('Embeddings shape:', embeddings.shape)
print('Dtype:           ', embeddings.dtype)


Extracting ProtBERT embeddings:   0%|          | 0/287 [00:00<?, ?it/s]

Embeddings shape: (2293, 1024)
Dtype:            float32


## 9) Sanity checks

In [9]:
assert embeddings.shape[0] == len(df), 'Row count mismatch'
assert not np.isnan(embeddings).any(), 'NaN in embeddings'
assert not np.isinf(embeddings).any(), 'Inf in embeddings'

print('Shape:    ', embeddings.shape)
print('Dtype:    ', embeddings.dtype)
print(f'Mean:     {embeddings.mean():.6f}')
print(f'Std:      {embeddings.std():.6f}')
print(f'Min:      {embeddings.min():.6f}')
print(f'Max:      {embeddings.max():.6f}')
print('Any NaN:  ', np.isnan(embeddings).any())
print('Any Inf:  ', np.isinf(embeddings).any())


Shape:     (2293, 1024)
Dtype:     float32
Mean:     0.001169
Std:      0.104909
Min:      -2.008961
Max:      5.207705
Any NaN:   False
Any Inf:   False


## 10) Encode labels and save arrays

Labels are integer-encoded using `LabelEncoder` with the same family ordering
as the ESM-2 notebooks (alphabetical by Pfam ID), ensuring consistent class
indices across all protein transformer comparisons.

Three parallel arrays saved in original row order.
Notebook 20 applies `train_test_split(random_state=SEED, test_size=0.2,
stratify=y)` -- identical to all prior protein notebooks.

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels       = le.fit_transform(df['family'])
family_names = df['family'].astype(str).values

print('Label shape:        ', labels.shape)
print('Classes (encoded):  ', le.classes_)
print('Counts per class:   ', dict(zip(*np.unique(labels, return_counts=True))))

sfx          = f'top{N_FAM}_per{PER_FAM}'
emb_path     = PROCESSED / f'protein_protbert_embeddings_{sfx}.npy'
labels_path  = PROCESSED / f'protein_protbert_labels_{sfx}.npy'
fnames_path  = PROCESSED / f'protein_protbert_family_names_{sfx}.npy'

np.save(emb_path,    embeddings.astype(np.float32))
np.save(labels_path, labels)
np.save(fnames_path, family_names)

print('Saved:', emb_path)
print('Saved:', labels_path)
print('Saved:', fnames_path)


Label shape:         (2293,)
Classes (encoded):   ['PF00001' 'PF00046' 'PF00069' 'PF00071' 'PF00076' 'PF00096' 'PF01352'
 'PF07686' 'PF12796' 'PF13853']
Counts per class:    {np.int64(0): np.int64(287), np.int64(1): np.int64(187), np.int64(2): np.int64(203), np.int64(3): np.int64(125), np.int64(4): np.int64(121), np.int64(5): np.int64(194), np.int64(6): np.int64(350), np.int64(7): np.int64(309), np.int64(8): np.int64(117), np.int64(9): np.int64(400)}
Saved: /home/dpratapa/Capstone/data/processed/protein_protbert_embeddings_top10_per400.npy
Saved: /home/dpratapa/Capstone/data/processed/protein_protbert_labels_top10_per400.npy
Saved: /home/dpratapa/Capstone/data/processed/protein_protbert_family_names_top10_per400.npy


## 11) Reload verification

In [11]:
X_check = np.load(emb_path)
y_check = np.load(labels_path)
f_check = np.load(fnames_path, allow_pickle=True)

assert X_check.shape == embeddings.shape
assert y_check.shape == labels.shape
assert f_check.shape == family_names.shape
assert not np.isnan(X_check).any()
assert (y_check == labels).all()

print('Reload check passed.')
print('Embedding file shape:', X_check.shape)
print('Labels file shape:   ', y_check.shape)
print('Family names shape:  ', f_check.shape)


Reload check passed.
Embedding file shape: (2293, 1024)
Labels file shape:    (2293,)
Family names shape:   (2293,)


## 12) Save summary JSON

In [12]:
label_counts = {int(k): int(v) for k, v in zip(*np.unique(labels, return_counts=True))}
family_counts = df['family'].value_counts().to_dict()

summary = {
    'notebook': '19_protein_protbert_embeddings',
    'model': MODEL_NAME,
    'model_size': '~420M parameters (BERT-large)',
    'pretrain_data': 'UniRef100 (~217M sequences)',
    'tokenizer_type': 'single amino acid, space-separated',
    'pooling_strategy': 'mean over body tokens (CLS and SEP excluded)',
    'rare_aa_handling': 'U, Z, O, B replaced with X before tokenization',
    'embedding_dim': int(embeddings.shape[1]),
    'n_sequences': int(embeddings.shape[0]),
    'n_families': int(df['family'].nunique()),
    'label_encoder_classes': le.classes_.tolist(),
    'label_counts': label_counts,
    'family_counts': {str(k): int(v) for k, v in family_counts.items()},
    'max_token_length': 1024,
    'batch_size_used': BATCH_SIZE,
    'device': DEVICE,
    'seed': SEED,
    'embedding_stats': {
        'mean': float(embeddings.mean()),
        'std':  float(embeddings.std()),
        'min':  float(embeddings.min()),
        'max':  float(embeddings.max()),
        'any_nan': bool(np.isnan(embeddings).any()),
        'any_inf': bool(np.isinf(embeddings).any()),
    },
    'output_files': {
        'embeddings':    str(emb_path),
        'labels':        str(labels_path),
        'family_names':  str(fnames_path),
    },
    'timestamp': pd.Timestamp.now().isoformat(),
}

summary_path = REPORTS / 'protein_protbert_embeddings_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved to:', summary_path)
print(json.dumps(summary, indent=2))


Summary saved to: /home/dpratapa/Capstone/reports/protein_protbert_embeddings_summary.json
{
  "notebook": "19_protein_protbert_embeddings",
  "model": "Rostlab/prot_bert",
  "model_size": "~420M parameters (BERT-large)",
  "pretrain_data": "UniRef100 (~217M sequences)",
  "tokenizer_type": "single amino acid, space-separated",
  "pooling_strategy": "mean over body tokens (CLS and SEP excluded)",
  "rare_aa_handling": "U, Z, O, B replaced with X before tokenization",
  "embedding_dim": 1024,
  "n_sequences": 2293,
  "n_families": 10,
  "label_encoder_classes": [
    "PF00001",
    "PF00046",
    "PF00069",
    "PF00071",
    "PF00076",
    "PF00096",
    "PF01352",
    "PF07686",
    "PF12796",
    "PF13853"
  ],
  "label_counts": {
    "0": 287,
    "1": 187,
    "2": 203,
    "3": 125,
    "4": 121,
    "5": 194,
    "6": 350,
    "7": 309,
    "8": 117,
    "9": 400
  },
  "family_counts": {
    "PF13853": 400,
    "PF01352": 350,
    "PF07686": 309,
    "PF00001": 287,
    "PF00069

## Next

Proceed to `20_protein_protbert_models.ipynb`:
- Loads `protein_protbert_embeddings_*.npy` and `protein_protbert_labels_*.npy`
- Applies `train_test_split(random_state=SEED, test_size=0.2, stratify=y)`
- Trains: Logistic Regression, SVM, Random Forest, XGBoost
- Reports: accuracy, precision (macro), recall (macro), F1 (macro), ROC-AUC (OvR)
- Compares against ESM-2 results from notebook 14
- Saves: CSV + JSON results, model `.pkl` files